###**Coleta dos Dados Bioclimáticos (2012-2024)**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Aponta para a RAIZ do projeto
%cd /content/drive/MyDrive/cafeicultura-varginha-analytics

In [ ]:
# coleta de dados bioclimáticos (2012-2024)  do INMET (Instituto Nacional de Meteorologia)

import os
import requests

# 1. Definindo intervalo de anos do projeto
anos = range(2012, 2025)
base_url = "https://portal.inmet.gov.br/uploads/dadoshistoricos"
pasta_raw = "data/raw"

print("Iniciando o download dos dados históricos do INMET (2012-2024)...\n")

# 2. Loop para baixar cada pacote anual
for ano in anos:
    url = f"{base_url}/{ano}.zip"
    caminho_zip = os.path.join(pasta_raw, f"{ano}.zip")

    if not os.path.exists(caminho_zip):
        print(f"Baixando {ano}.zip...")
        resposta = requests.get(url, stream=True)

        if resposta.status_code == 200:
            with open(caminho_zip, "wb") as f:
                for chunk in resposta.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✔ {ano}.zip salvo com sucesso.")
        else:
            print(f"✖ Falha ao baixar {ano}.zip (Status Code: {resposta.status_code})")
    else:
        print(f"➜ Arquivo {ano}.zip já existe em {pasta_raw}.")

print("\nDownload concluído!")

In [ ]:
# Filtrando e extraindo apenas a estação de Varginha-MG

import zipfile

estacao_varginha = "VARGINHA"
pasta_extraida = "data/raw/varginha_raw"
os.makedirs(pasta_extraida, exist_ok=True)

for ano in anos:
    caminho_zip = os.path.join(pasta_raw, f"{ano}.zip")

    if os.path.exists(caminho_zip):
        with zipfile.ZipFile(caminho_zip, "r") as zip_ref:
            # Procura no zip pelo arquivo CSV que contém VARGINHA no nome
            arquivos = zip_ref.namelist()
            arquivo_varginha = [
                f for f in arquivos if estacao_varginha in f.upper()
            ]

            if arquivo_varginha:
                # Extrai apenas o CSV de Varginha
                zip_ref.extract(arquivo_varginha[0], pasta_extraida)
                print(f"✔ Dados de Varginha ({ano}) extraídos com sucesso.")
            else:
                print(f"⚠️ Estação de Varginha não encontrada no pacote de {ano}.")

In [ ]:
# Unificando os arquivos anuais em um único DataFrame

import glob
import pandas as pd

# 1. Localiza todos os arquivos CSV extraídos da pasta de Varginha
arquivos_csv = glob.glob(
    "data/raw/varginha_raw/**/*.CSV", recursive=True
) + glob.glob("data/raw/varginha_raw/**/*.csv", recursive=True)

lista_dfs = []

# 2. Lê cada arquivo aplicando os ajustes necessários do INMET
for arquivo in sorted(arquivos_csv):
    df = pd.read_csv(
        arquivo, sep=";", skiprows=8, encoding="latin-1", decimal=","
    )
    lista_dfs.append(df)

# 3. Une todos os anos em um único DataFrame
df_varginha = pd.concat(lista_dfs, ignore_index=True)

print(
    f"✔ Dados consolidados! Total de linhas carregadas: {len(df_varginha):,}"
)
df_varginha.head(3)

###**Unificar os arquivos anuais em um único DataFrame**

In [24]:
import glob
import pandas as pd

# 1. Localiza todos os arquivos CSV extraídos da pasta de Varginha
arquivos_csv = glob.glob(
    "data/raw/varginha_raw/**/*.CSV", recursive=True
) + glob.glob("data/raw/varginha_raw/**/*.csv", recursive=True)

lista_dfs = []

# 2. Lê cada arquivo aplicando os ajustes necessários do INMET
for arquivo in sorted(arquivos_csv):
    df = pd.read_csv(
        arquivo, sep=";", skiprows=8, encoding="latin-1", decimal=","
    )
    lista_dfs.append(df)

# 3. Une todos os anos em um único DataFrame
df_varginha = pd.concat(lista_dfs, ignore_index=True)

print(
    f"✔ Dados consolidados! Total de linhas carregadas: {len(df_varginha):,}"
)
df_varginha.head(3)

,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (KJ/m²),RADIACAO GLOBAL (Kj/m²),TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),...,temp_max_c,temp_min_c,umidade_rel_pct,hora_limpa,data_hora,ano,mes,dia,hora_num,dia_do_ano
0,0.6,904.5,904.5,904.0,NaN,NaN,18.9,19.1,18.9,96.0,...,19.9,19.6,96.0,00:00,2012-01-01 00:00:00,2012,1,1,0,1
1,2.2,904.7,904.7,904.5,NaN,NaN,18.7,18.9,18.7,97.0,...,19.6,19.2,97.0,01:00,2012-01-01 01:00:00,2012,1,1,1,1
2,0.6,904.2,904.7,904.1,NaN,NaN,19.0,19.0,18.7,97.0,...,19.5,19.2,97.0,02:00,2012-01-01 02:00:00,2012,1,1,2,1


###**Padronização de colunas, formatação de Data/Hora e salvamento do dataset unificado.**
1. Limpeza de colunas e criação da variável temporal


In [25]:
# 1. Mapeamento de renomeação
colunas_renomear = {
    'Data': 'data',
    'DATA (YYYY-MM-DD)': 'data',
    'Hora UTC': 'hora',
    'HORA (UTC)': 'hora',
    'PRECIPITAÇÃO TOTAL, HORÁRIA (mm)': 'precipitacao_mm',
    'PRECIPITACAO TOTAL, HORARIA (mm)': 'precipitacao_mm',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'temp_ar_c',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (Â°C)': 'temp_ar_c',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temp_max_c',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temp_min_c',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_rel_pct',
    'VENTO, VELOCIDADE (m/s)': 'vento_velocidade_ms',
}

df_varginha = df_varginha.rename(columns=colunas_renomear)

# 2. Consolida colunas duplicadas criadas pelo rename
df_varginha = df_varginha.T.groupby(level=0).first().T

# 3. Trata a coluna 'hora' e cria 'data_hora'
df_varginha['hora_limpa'] = (
    df_varginha['hora']
    .astype(str)
    .str.replace(' UTC', '')
    .str.zfill(4)
    .str[:2]
    + ':00'
)

df_varginha['data_hora'] = pd.to_datetime(
    df_varginha['data'].astype(str) + ' ' + df_varginha['hora_limpa'],
    format='mixed',
    errors='coerce',
)

# 4. Adiciona colunas temporais separadas
df_varginha['ano'] = df_varginha['data_hora'].dt.year
df_varginha['mes'] = df_varginha['data_hora'].dt.month
df_varginha['dia'] = df_varginha['data_hora'].dt.day
df_varginha['hora_num'] = df_varginha['data_hora'].dt.hour
df_varginha['dia_do_ano'] = df_varginha['data_hora'].dt.dayofyear

# 5. Ordena e salva o dataset bruto consolidado
df_varginha = df_varginha.sort_values('data_hora').reset_index(drop=True)
caminho_saida = 'data/raw/varginha_2012_2024_consolidado.csv'
df_varginha.to_csv(caminho_saida, index=False, encoding='utf-8')

print(f'✔ Dataset consolidado salvo em: {caminho_saida}')

df_varginha.head(3)


,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (KJ/m²),RADIACAO GLOBAL (Kj/m²),TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),...,temp_max_c,temp_min_c,umidade_rel_pct,hora_limpa,data_hora,ano,mes,dia,hora_num,dia_do_ano
0,0.6,904.5,904.5,904.0,NaN,NaN,18.9,19.1,18.9,96.0,...,19.9,19.6,96.0,00:00,2012-01-01 00:00:00,2012,1,1,0,1
1,2.2,904.7,904.7,904.5,NaN,NaN,18.7,18.9,18.7,97.0,...,19.6,19.2,97.0,01:00,2012-01-01 01:00:00,2012,1,1,1,1
2,0.6,904.2,904.7,904.1,NaN,NaN,19.0,19.0,18.7,97.0,...,19.5,19.2,97.0,02:00,2012-01-01 02:00:00,2012,1,1,2,1


2. Versionamento no Git


In [21]:
!git add 01_coleta_dados.ipynb
!git commit -m "feat: adiciona coleta e consolidacao dos dados do INMET"
!git push origin main

[main a81b5cf] feat: adiciona coleta e consolidacao dos dados do INMET
 1 file changed, 1 insertion(+)
 create mode 100644 01_coleta_dados.ipynb
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 5.52 KiB | 565.00 KiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/karine94/cafeicultura-varginha-analytics.git
   6ce6224..a81b5cf  main -> main


####**Tratamento de Nulos e Validação dos Dados Meteorológicos**

In [22]:
# Mapeando e identificando valores nulos e as falhas de leitura dos sensores

import numpy as np

# 1. Substitui a convenção do INMET (-9999) por NaN do NumPy
df_varginha = df_varginha.replace([-9999, -9999.0, "-9999"], np.nan)

# 2. Preenche nulos na precipitação com 0 (ausência de chuva registrada)
if "precipitacao_mm" in df_varginha.columns:
    df_varginha["precipitacao_mm"] = df_varginha["precipitacao_mm"].fillna(0)

# 3. Calcula o percentual de dados ausentes por coluna
nulos_pct = (df_varginha.isna().sum() / len(df_varginha)) * 100

print("=== Percentual de Dados Ausentes por Coluna (%) ===")
print(nulos_pct.round(2))

/tmp/ipykernel_4037/3405589994.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_varginha = df_varginha.replace([-9999, -9999.0, "-9999"], np.nan)


=== Percentual de Dados Ausentes por Coluna (%) ===
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          13.90
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)      3.42
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)            3.42
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)           3.42
RADIACAO GLOBAL (KJ/m²)                                   69.06
RADIACAO GLOBAL (Kj/m²)                                   79.45
TEMPERATURA DO PONTO DE ORVALHO (°C)                       3.46
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)           3.48
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)           3.48
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                   3.47
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                   3.47
Unnamed: 19                                              100.00
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                       3.42
VENTO, RAJADA MAXIMA (m/s)                                 3.43
VENTO, VELOCIDADE HORARIA (m/s)                     

In [26]:
# Salva o CSV atualizado com os NaNs devidamente mapeados
df_varginha.to_csv("data/raw/varginha_2012_2024_consolidado.csv", index=False)

In [ ]:
!git add 01_coleta_dados.ipynb
!git commit -m "docs: atualiza mapeamento de nulos no notebook 01"
!git push origin main